# Fine-Tuning with LoRA - Nexus-LLM

This notebook demonstrates how to fine-tune a language model using LoRA (Low-Rank Adaptation) with Nexus-LLM.

## What is LoRA?

LoRA freezes the pre-trained model weights and injects trainable rank decomposition matrices into each layer, dramatically reducing the number of trainable parameters while maintaining quality.

## 1. Prepare the Dataset

In [ ]:
from nexus_llm.training import DatasetLoader, DataPreprocessor

# Load dataset from JSONL
loader = DatasetLoader()
dataset = loader.load_jsonl(
    path="data/train.jsonl",
    validation_split=0.1,
    seed=42,
)

print(f"Training samples: {len(dataset.train)}")
print(f"Validation samples: {len(dataset.val)}")

# Inspect a sample
sample = dataset.train[0]
print(f"\nSample instruction: {sample['instruction'][:100]}...")
print(f"Sample output: {sample['output'][:100]}...")

In [ ]:
# Preprocess and format for training
preprocessor = DataPreprocessor(
    format="chatml",           # Chat format: chatml, alpaca, sharegpt
    max_length=2048,           # Max sequence length
    truncate=True,
)

train_data = preprocessor.process(dataset.train)
val_data = preprocessor.process(dataset.val)

print(f"Processed training samples: {len(train_data)}")
print(f"Average token length: {train_data.avg_length:.0f}")

## 2. Configure LoRA

In [ ]:
from nexus_llm.training import LoRAConfig

lora_config = LoRAConfig(
    r=16,                        # LoRA rank (try 8, 16, 32, 64)
    lora_alpha=32,               # Scaling factor (typically 2x rank)
    target_modules=[
        "q_proj",               # Query projection
        "k_proj",               # Key projection
        "v_proj",               # Value projection
        "o_proj",               # Output projection
    ],
    lora_dropout=0.05,           # Dropout for regularization
    bias="none",                # No bias training
    task_type="CAUSAL_LM",
)

# Print parameter counts
from nexus_llm.training import ParameterCounter
counter = ParameterCounter("nexus-7b-chat", lora_config)
print(f"Total model parameters: {counter.total_params:,}")
print(f"Trainable LoRA parameters: {counter.trainable_params:,}")
print(f"Trainable ratio: {counter.trainable_ratio:.2%}")

## 3. Training

In [ ]:
from nexus_llm.training import Trainer

trainer = Trainer(
    base_model="nexus-7b-chat",
    lora_config=lora_config,
    output_dir="./checkpoints/lora-tutorial",
    training_args={
        "num_train_epochs": 3,
        "per_device_train_batch_size": 4,
        "gradient_accumulation_steps": 4,
        "learning_rate": 2e-4,
        "lr_scheduler_type": "cosine",
        "warmup_ratio": 0.03,
        "weight_decay": 0.01,
        "fp16": True,
        "gradient_checkpointing": True,
        "logging_steps": 10,
        "eval_strategy": "steps",
        "eval_steps": 50,
        "save_strategy": "steps",
        "save_steps": 50,
        "save_total_limit": 3,
        "report_to": "tensorboard",
    },
)

# Start training
trainer.train(
    train_dataset=train_data,
    eval_dataset=val_data,
)

## 4. View Training Metrics

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt

history = trainer.get_training_history()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(history["step"], history["train_loss"], label="Train Loss")
axes[0].plot(history["step"], history["eval_loss"], label="Eval Loss")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training & Validation Loss")
axes[0].legend()

# Learning rate schedule
axes[1].plot(history["step"], history["learning_rate"])
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Learning Rate")
axes[1].set_title("Learning Rate Schedule")

plt.tight_layout()
plt.show()

## 5. Evaluate the Fine-Tuned Model

In [ ]:
from nexus_llm import InferenceEngine, Conversation

# Load the fine-tuned model
ft_engine = InferenceEngine(
    model_name="nexus-7b-chat",
    adapter_path="./checkpoints/lora-tutorial/final_adapter",
    device="auto",
)

# Compare with base model
base_engine = InferenceEngine(
    model_name="nexus-7b-chat",
    device="auto",
)

test_prompt = "Explain the key principles of our product's design philosophy."

base_response = base_engine.chat(test_prompt)
ft_response = ft_engine.chat(test_prompt)

print("Base model response:")
print(base_response.text[:300])
print("\n" + "="*50 + "\n")
print("Fine-tuned model response:")
print(ft_response.text[:300])

## 6. Save and Merge

In [ ]:
# Save just the LoRA adapter (small, portable)
trainer.save_adapter("./my_lora_adapter")
print("LoRA adapter saved!")

# Optionally: merge LoRA weights into the base model
# This creates a standalone model that doesn't need the adapter
trainer.merge_and_save("./models/my-merged-model")
print("Merged model saved!")

## Tips for Better Fine-Tuning

1. **Data quality matters more than quantity** - Clean, well-formatted examples beat large noisy datasets
2. **Start with a low learning rate** - 2e-4 is a good starting point for LoRA
3. **Use gradient checkpointing** - Saves memory at the cost of ~20% slower training
4. **Monitor eval loss** - Stop training when eval loss stops improving
5. **Experiment with LoRA rank** - Higher rank = more capacity but risk of overfitting